In [30]:
import findspark
import numpy as np
import pandas as pd
import subprocess
import joblib
import time

from tqdm.auto import tqdm

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Optimized for MSI GF63") \
    .master("local[10]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .config("spark.default.parallelism", "12") \
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "300s") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.python.worker.reuse", "true") \
    .getOrCreate()

spark

In [4]:
print(
    subprocess.check_output(
        "nvidia-smi",
        shell=True
    ).decode()
)

Sat May 16 05:23:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 582.53                 Driver Version: 582.53         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro M1000M                WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A    0C    P8            N/A  /  200W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
DATA_PATH = "Data/Feature_Engineered_Data"

In [6]:
train_df = spark.read.parquet(f"{DATA_PATH}/train")
test_df  = spark.read.parquet(f"{DATA_PATH}/test")

In [7]:
print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 6987064
Test: 1747953


In [8]:
test_df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 |-- Wind_Direction: string (nullable = true)
 |-- Wind_Speed(mph): double (nullable = true)
 |-- Precipitation(in): double (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Amenity: integer (nullable = true)
 |-- Bump: integer (nullable = true)
 |-- Crossing: integer (nullable = true)
 |-- Give_Way: integer (nullable = true)
 |-- Junction: integer (nullable = true)
 |-- No_Exit: integer 

In [9]:
train_df = train_df.withColumn(
    "label",
    col("Severity")
)

test_df = test_df.withColumn(
    "label",
    col("Severity")
)

In [10]:
train_df = train_df.select(
    "features",
    "label",
    "weight"
)

test_df = test_df.select(
    "features",
    "label",
    "weight"
)

train_df.cache()
test_df.cache()

print(
    "Train Samples:",
    train_df.count()
)

print(
    "Test Samples:",
    test_df.count()
)

Train Samples: 6987064
Test Samples: 1747953


In [11]:
def spark_to_numpy_stream(df):

    total_rows=df.count()

    X=[]
    y=[]
    w=[]

    for row in tqdm(
            df.toLocalIterator(),
            total=total_rows,
            desc="Converting",
            unit="rows"
    ):

        X.append(
            row["features"].toArray()
        )

        y.append(
            row["label"]
        )

        w.append(
            row["weight"]
        )

    return (
        np.array(X),
        np.array(y),
        np.array(w)
    )

In [12]:
X_train,y_train,w_train = (
    spark_to_numpy_stream(
        train_df
    )
)
print(
    "Train Data Shapes:",
    X_train.shape,
    y_train.shape,
    w_train.shape
)

Converting:   0%|          | 0/6987064 [00:00<?, ?rows/s]

Train Data Shapes: (6987064, 39) (6987064,) (6987064,)


In [13]:
X_test,y_test,w_test = (
    spark_to_numpy_stream(
        test_df
    )
)
print(
    "Test Data Shapes:",
    X_test.shape,
    y_test.shape,
    w_test.shape
)

Converting:   0%|          | 0/1747953 [00:00<?, ?rows/s]

Test Data Shapes: (1747953, 39) (1747953,) (1747953,)


In [34]:
model=lgb.LGBMClassifier(

    objective='multiclass',

    num_class=4,

    n_estimators=900,

    learning_rate=0.05,

    max_depth=8,

    num_leaves=64,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    device='gpu',

    gpu_platform_id=1,

    gpu_device_id=0
)

In [36]:
start=time.time()

print(
    "Training Started..."
)

model.fit(

    X_train,

    y_train,

    sample_weight=w_train,

    eval_set=[
        (X_test,y_test)
    ]
)
end=time.time()

print(
    f"Training Time: {(end-start)/60:.2f} minutes"
)

Training Started...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2344
[LightGBM] [Info] Number of data points in the train set: 6987064, number of used features: 38
[LightGBM] [Info] Using requested OpenCL platform 1 device 0
[LightGBM] [Info] Using GPU Device: Quadro M1000M, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 17 dense feature groups (133.27 MB) transferred to GPU in 0.252134 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -1.384576
[LightGBM] [Info] Start training from score -1.386931
[LightGBM] [Info] Start training from score -1.387355
[LightGBM] [Info] Start training from score -1.386318
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training Time: 48.00 minutes


In [37]:
start=time.time()

print(
    "Predicting..."
)

y_pred=model.predict(
    X_test
)

end=time.time()

print(
    f"Prediction Time: {end-start:.2f} sec"
)

Predicting...


C:\Users\grand\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Prediction Time: 641.62 sec


In [38]:
acc=accuracy_score(
    y_test,
    y_pred
)

print(
    "Accuracy:",
    acc
)

Accuracy: 0.6413868107437671


In [39]:
f1=f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print(
    "Weighted F1:",
    f1
)

Weighted F1: 0.7054202370399095


In [40]:
print(

    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           1       0.10      0.92      0.18     14604
           2       0.97      0.61      0.75   1412570
           3       0.48      0.74      0.59    270299
           4       0.14      0.81      0.23     50480

    accuracy                           0.64   1747953
   macro avg       0.42      0.77      0.44   1747953
weighted avg       0.86      0.64      0.71   1747953



In [41]:
cm=confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[ 13479    322    581    222]
 [102244 865347 212908 232071]
 [ 19131  20832 201340  28996]
 [   717   6451   2364  40948]]


In [ ]:
joblib.dump(

    model,

    "Models/lightgbm_model.pkl"
    
)

print(
    "Model Saved"
)

Model Saved
